In [1]:
!git clone https://github.com/GyanRout/Major-Projects.git
%cd Major-Projects
!git pull

Cloning into 'Major-Projects'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 54 (delta 12), reused 39 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 498.74 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/Major-Projects
Already up to date.


In [2]:
!pip install -r /content/Major-Projects/dp_recommender_project/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.9/308.9 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.7 MB/s eta 0:00:00


In [3]:
import sys

# Adding project root directory to Python's path
project_root = '/content/Major-Projects/dp_recommender_project'
if project_root not in sys.path:
    sys.path.append(project_root)

In [4]:
from src.data_utils import Loader
from src.model_defs import RecommenderModel
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader
import json

In [5]:
train_df_path = '/content/Major-Projects/dp_recommender_project/data/train_data.csv'
test_df_path = '/content/Major-Projects/dp_recommender_project/data/test_data.csv'
movie_idx_path = '/content/Major-Projects/dp_recommender_project/data/movie_to_index.json'
user_idx_path = '/content/Major-Projects/dp_recommender_project/data/user_to_index.json'

BATCH_SIZE = 2048
LEARNING_RATE = 0.01
with open(user_idx_path,'r') as f:
  NUM_USER = len(json.load(f))

with open(movie_idx_path,'r') as f:
  NUM_MOVIE = len(json.load(f))

train_df = pd.read_csv(train_df_path)
test_df = pd.read_csv(test_df_path)

train_dataset = Loader(train_df)
test_dataset = Loader(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

In [6]:
from opacus import PrivacyEngine

NOISE_MULTIPLIER=1.2
MAX_GRAD_NORM=1.0

model = RecommenderModel(num_user=NUM_USER,num_item=NUM_MOVIE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

privacy_engine = PrivacyEngine()

model, optimizer, train_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=train_loader,
    noise_multiplier=NOISE_MULTIPLIER,
    max_grad_norm=MAX_GRAD_NORM,
)

print("The Privacy Engine is successfully attached!")

The Privacy Engine is successfully attached!


/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [7]:
from opacus.utils.batch_memory_manager import BatchMemoryManager

EPOCH = 15
MAX_BATCH_SIZE = 128
TARGET_DELTA = 1e-5

loss_fn = nn.HuberLoss()

model.train()

for epoch in range(EPOCH):
  epoch_loss = 0.0
  batch_count = 0

  with BatchMemoryManager(
      data_loader=train_loader,
      max_physical_batch_size=MAX_BATCH_SIZE,
      optimizer=optimizer
  ) as memory_safe_loader:

    for user_id, item_id, ratings in memory_safe_loader:
      user_id = user_id.to(device)
      item_id = item_id.to(device)
      ratings = ratings.to(device).view(-1,1)

      optimizer.zero_grad()
      predictions = model(user_id, item_id)
      loss = loss_fn(predictions, ratings)
      loss.backward()
      optimizer.step()

      epoch_loss += loss.item()
      batch_count += 1

  epsilon = privacy_engine.get_epsilon(TARGET_DELTA)
  avg_loss= epoch_loss/batch_count

  print(f"Epoch: {epoch+1}/{EPOCH} | Average Loss: {avg_loss:.4f} | Privacy Budget: {epsilon:.2f}")

sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.


Epoch: 1/15 | Average Loss: 0.4901 | Privacy Budget: 0.84
Epoch: 2/15 | Average Loss: 0.4013 | Privacy Budget: 1.09
Epoch: 3/15 | Average Loss: 0.3658 | Privacy Budget: 1.29
Epoch: 4/15 | Average Loss: 0.3489 | Privacy Budget: 1.46
Epoch: 5/15 | Average Loss: 0.3356 | Privacy Budget: 1.62
Epoch: 6/15 | Average Loss: 0.3315 | Privacy Budget: 1.76
Epoch: 7/15 | Average Loss: 0.3236 | Privacy Budget: 1.89
Epoch: 8/15 | Average Loss: 0.3225 | Privacy Budget: 2.01
Epoch: 9/15 | Average Loss: 0.3202 | Privacy Budget: 2.13
Epoch: 10/15 | Average Loss: 0.3125 | Privacy Budget: 2.25
Epoch: 11/15 | Average Loss: 0.3121 | Privacy Budget: 2.35
Epoch: 12/15 | Average Loss: 0.3091 | Privacy Budget: 2.46
Epoch: 13/15 | Average Loss: 0.3082 | Privacy Budget: 2.56
Epoch: 14/15 | Average Loss: 0.3044 | Privacy Budget: 2.66
Epoch: 15/15 | Average Loss: 0.3034 | Privacy Budget: 2.75


In [8]:
clean_state_dict = model._module.state_dict()

save_path = '/content/Major-Projects/dp_recommender_project/src/private_recommender.pth'
torch.save(clean_state_dict, save_path)

print(f"Model parameters successfully saved to {save_path}")

Model parameters successfully saved to /content/Major-Projects/dp_recommender_project/src/private_recommender.pth
